# 實作 1: 儲存及套用使用者的偏好

## 建立 Store 

In [5]:
from langgraph.store.memory import InMemoryStore
store = InMemoryStore()

# User 123 偏好
store.put(
    namespace = ("users", "user_123", "preferences"),
    key = "communication",
    value = {
        "preferred_language": "台灣繁體中文",
        "preferred_answer_style": "精簡、直接"
    }
)

# User 456 偏好
store.put(
    namespace = ("users", "user_456", "preferences"),
    key = "communication",
    value = {
        "preferred_language": "English",
        "preferred_answer_style": "detailed, step-by-step"
    }
)

## 建立 Static runtime context 的 schema (資料結構)

In [6]:
from dataclasses import dataclass

@dataclass
class UserStaticContext:
    user_id: str

## 建立 System Prompt

In [7]:
BASE_SYSTEM_PROMPT = """
你是一位 ERP 的教師，說明與解釋 ERP 的概念與商業流程。
你不會回答系統操作的流程問題。

回答必須正確、清楚。

如果訓練資料中沒有明確的答案，請說明「我不確定」或「我不知道」，不要亂猜。
如果是臆測的答案，請在回答中說明「這是我的臆測」。

"""

## 建立 Middleware function

In [ ]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain.messages import SystemMessage

@dynamic_prompt
def user_dynamic_prompt(request: ModelRequest[UserStaticContext]) -> str | SystemMessage:
    """
    此 Middleware 會根據目前使用者的偏好，動態修改 System Prompt。

    流程如下：
    1. 從 runtime context 取得使用者 ID。
    2. 從 Memory Store 讀取該使用者的溝通偏好。
    3. 取得偏好的回覆語言與回答風格。
    4. 將偏好設定附加至目前的 System Prompt。
    5. 回傳更新後的 SystemMessage。

    :param request: ModelRequest 物件，包含 runtime context 與 store。
    :return: 更新後的 SystemMessage，包含使用者偏好設定。

    """
    # 1. 從 modelRequest 中取得 static runtime context 中的 user_id
    # UserStaticContext is the context scheme
    user_id = request.runtime.context.user_id

    # 2. 從 modelRequest 中取得目前的 store
    store = request.runtime.store

    # 3. 操作 Memory Store，去該 user_id 的 namespace 取得偏好資料
    # Namespace format: ("users", user_id, "preferences")

    comm_pref = (
        store.get(
        namespace = ("users", user_id, "preferences"),
        key = "communication")
        if store is not None else None
    )

    # get the preference 
    preferences = comm_pref.value if comm_pref is not None else {}

    # 4. 製作回覆的語言與風格的文字說明
    
    style_instruction = f"請使用 {preferences.get('preferred_language', 'French')} 回覆，並且回答的內容要 {preferences.get('preferred_answer_style', '精簡、直接')}。"

    # 5. 從 modelRequest 中取得目前的 System Prompt
    current_system_prompt = request.system_prompt if request.system_prompt is not None else ""

    # 6. 將這些文字說明加入到 System Prompt 中，並回傳新的 System Prompt
    dynamic_system_prompt = current_system_prompt.strip() + "\n" + style_instruction

    # 僅供課堂觀察；正式環境可改用適當的 logging
    print("=== Current System Prompt ===")
    print(current_system_prompt)
    print("=== Dynamic System Prompt ===")
    print(dynamic_system_prompt)

    return SystemMessage(content=dynamic_system_prompt)

## 建立 Agent

In [9]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    system_prompt=BASE_SYSTEM_PROMPT,
    tools=[],
    context_schema=UserStaticContext, # User's static runtime context schema
    store=store,  # Memory Store
    middleware=[user_dynamic_prompt] # Middleware functions
)

## 比較兩位使用者

In [10]:
# First user's question
user_123_question = {
    "messages": [
        { "role": "user", "content": "請說明什麼是請購到採購的商業流程。" }
    ]
}

In [11]:
# Invoke the agent with the first user's question
result_123 = agent.invoke(
    user_123_question,
    context=UserStaticContext(user_id="user_123")
)

=== Current System Prompt ===

你是一位 ERP 的教師，說明與解釋 ERP 的概念與商業流程。
你不會回答系統操作的流程問題。

回答必須正確、清楚。

如果訓練資料中沒有明確的答案，請說明「我不確定」或「我不知道」，不要亂猜。
如果是臆測的答案，請在回答中說明「這是我的臆測」。


=== Dynamic System Prompt ===

你是一位 ERP 的教師，說明與解釋 ERP 的概念與商業流程。
你不會回答系統操作的流程問題。

回答必須正確、清楚。

如果訓練資料中沒有明確的答案，請說明「我不確定」或「我不知道」，不要亂猜。
如果是臆測的答案，請在回答中說明「這是我的臆測」。


請使用 台灣繁體中文 回覆，並且回答的內容要 精簡、直接。


In [13]:
from pprint import pp

pp(result_123["messages"][-1].content)

('以下是請購到採購的商業流程的概念性說明（高階流程，非系統操作步驟）：\n'
 '\n'
 '- 定義\n'
 '  - 請購（Purchase Requisition, '
 'PR）：內部單位提出需購買的物料或服務需求，包含品項、數量、預計交期、品質等，並分派成本中心與預算。\n'
 '  - 採購單（Purchase Order, PO）：對外與供應商簽約的正式訂單，載明供應商、品項、數量、價格、交貨日期、條件等。PR 通常會轉換成 '
 'PO。\n'
 '\n'
 '- 流程步驟（高階概要）\n'
 '  1) 需求與請購提出\n'
 '     - 部門提出需求，填寫 PR，包含物料/服務、數量、交期、品質要求、預算科目等。\n'
 '  2) 審核與採購策略\n'
 '     - 進行預算核可、需求有效性審查；決定是否需要比價、招標、或適用現有合約。\n'
 '  3) 從請購到採購單\n'
 '     - 將通過審核的 PR 轉換為對供應商可接受的 PO；確定供應商、價格、交貨條件等。\n'
 '  4) 供應商選擇與合約管理\n'
 '     - 如需要，進行招標/比價，選定供應商；若已有合約，依合約條款下單。\n'
 '  5) 下單通知與供應商回覆\n'
 '     - 向供應商下訂並待回覆（確認交貨時間、可用性等）。\n'
 '  6) 收貨與驗收\n'
 '     - 物料到貨，進行數量與品質驗收，完成入庫或服務完成的驗收。\n'
 '  7) 計價與對帳（發票處理前置）\n'
 '     - 對應的 PO、收貨紀錄與發票進行對帳（常見三方對帳：PO、收貨、發票），產生會計分錄。\n'
 '  8) 付款與結案\n'
 '     - 核准付款、結案，並進行供應商績效評估與報告。\n'
 '\n'
 '- 核心概念與要點\n'
 '  - 資料流：從 PR 流向 PO，涉及物料主檔、供應商資料、價格、交期、成本科目等。\n'
 '  - 控制與治理：預算控管、審核權限、授權分離、必要時的競爭性比價或招標、以及合約遵循。\n'
 '  - 成本與會計影響：與庫存、應付帳款、現金流、成本中心等相關的會計分錄與報表。\n'
 '\n'
 '如果需要，我可以用一個簡單的案例來說明各步驟如何在企業內部互動。')


In [14]:
# User_456's question
user_456_question = {
    "messages": [
        { "role": "user", "content": "請說明什麼是請購到採購的商業流程。" }
    ]
}   

In [15]:
result_456 = agent.invoke(
    user_456_question,
    context=UserStaticContext(user_id="user_456")
)  

=== Current System Prompt ===

你是一位 ERP 的教師，說明與解釋 ERP 的概念與商業流程。
你不會回答系統操作的流程問題。

回答必須正確、清楚。

如果訓練資料中沒有明確的答案，請說明「我不確定」或「我不知道」，不要亂猜。
如果是臆測的答案，請在回答中說明「這是我的臆測」。


=== Dynamic System Prompt ===

你是一位 ERP 的教師，說明與解釋 ERP 的概念與商業流程。
你不會回答系統操作的流程問題。

回答必須正確、清楚。

如果訓練資料中沒有明確的答案，請說明「我不確定」或「我不知道」，不要亂猜。
如果是臆測的答案，請在回答中說明「這是我的臆測」。


請使用 English 回覆，並且回答的內容要 detailed, step-by-step。


In [17]:
print (result_456["messages"][-1].content)

Here is a detailed, step-by-step explanation of the end-to-end business process commonly called " procure-to-pay" (P2P) or, in your wording, from purchase requisition (PR) to procurement (PO and beyond). This describes the typical flow in an ERP-enabled environment, focusing on concepts and the data/documents involved rather than system-specific操作 steps.

1) Demand identification and specification
- Who: An internal user or department.
- What happens: A need is identified for materials or services. The requester defines what is required (material/service), quantity, desired delivery date, quality/specifications, and any special handling.
- Data touched: item number or description, unit of measure, required date, plant/location, and any service scope if applicable.

2) Create and capture a purchase requisition (PR)
- Purpose: Formalize the internal request in the ERP system and establish a traceable, auditable record.
- What goes into a PR: material number (or service description), quan